# CS604 Colab launcher — Ana: real zoom boxes for the slides (no training)

Runs the exact path of `src.predict --zoom` on the 50,000 all-class holdout photos and keeps each photo's box (`scripts/zoom_boxes.py`). It needs a checkpoint with a labelled holdout, so it uses the all-class runs and not the finals, which trained on all 45 photos per class.

**Before running:** `scripts/zoom_boxes.py`, `scripts/plot_zoom_boxes.py` and this notebook must be pushed to `ana-branch`.

Outputs in `CS604_RUNS/zoom_boxes/<model>/`: `summary.json`, `scan.csv`, `candidates.json`, `candidates/` and `sheet_<bucket>.png`. The sheets show class IDs here because the species-name file is not in the repo.

About an hour on an A100, mostly the data download and reading the holdout at twice the resolution. Use **Runtime → Run all**.

**Afterwards, on your laptop:** download the output folder, look at the sheets, and draw the slide figure from the ids you pick (no GPU):

```bash
python scripts/plot_zoom_boxes.py --in_dir <folder> --sheets
python scripts/plot_zoom_boxes.py --in_dir <folder> --picks <id> <id> <id> --out figures/zoom_boxes
```

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
PILOT_DATA_DIR = "/content/pilot_data"
# Van's full FT 448 trained with the all-class holdout (notebooks/colab_full-ft_448px_task1-final_all-10000-classes.ipynb
# on van-allclass-ft448). The run folder is read from its arm record; or set RUN_DIR yourself.
ARM_RECORD = f"{DRIVE_OUT}/ablation_allclass_holdout_full_ft_448/arms/A_all_classes_144min.json"
RUN_DIR = None
CROP_PCT = 0.9  # Van's zoom setting at 448
OUT_NAME = "zoom_boxes/full_ft_448"
# LoRA instead (matches figures/zoom_only_holdout):
# RUN_DIR = f"{DRIVE_OUT}/ablation_allclass_holdout_lora_no_extras/runs/A_all_classes_144min/lora_20261026_123804"
# CROP_PCT = 0.75
# OUT_NAME = "zoom_boxes/lora_336"
# End the runtime after the last cell, or as soon as a step fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and find the checkpoint

If this cell errors on `CS604_RUNS`, add a shortcut with that name to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path
import json

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
if RUN_DIR is None:
    assert Path(ARM_RECORD).is_file(), f"no {ARM_RECORD}: has Van's all-class run finished? Otherwise set RUN_DIR"
    RUN_DIR = json.loads(Path(ARM_RECORD).read_text())["run_dir"]
assert (Path(RUN_DIR) / "last.pt").is_file() and (Path(RUN_DIR) / "config.yaml").is_file(), (
    f"no last.pt / config.yaml in {RUN_DIR}")
print("checkpoint found:", RUN_DIR)

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 5. Rebuild the holdout split, scan it and draw the contact sheets

In [ ]:
import subprocess
import sys

from scripts.pilot_sweep import build_pilot_root

# The split both all-class runs trained on: all 10,000 classes, 5 holdout photos each, seed 0
PILOT_ROOT, info = build_pilot_root(DATA_DIR, PILOT_DATA_DIR, 10000, 5, 0)
print(info)
OUT = Path(DRIVE_OUT) / OUT_NAME
proc = subprocess.run([sys.executable, "-u", "scripts/zoom_boxes.py", "--run_dir", RUN_DIR,
                       "--pilot_root", str(PILOT_ROOT), "--out_dir", str(OUT), "--crop_pct", str(CROP_PCT)])
if proc.returncode != 0:
    end_runtime(f"zoom scan failed (exit {proc.returncode})")
    raise SystemExit("zoom scan failed")
proc = subprocess.run([sys.executable, "-u", "scripts/plot_zoom_boxes.py", "--in_dir", str(OUT), "--sheets"])
if proc.returncode != 0:
    end_runtime(f"contact sheets failed (exit {proc.returncode})")
    raise SystemExit("contact sheets failed")
print(json.dumps(json.loads((OUT / "summary.json").read_text()), indent=2))

## 6. Results

- `top1_plain` and `top1_zoom` are this checkpoint's accuracy on the holdout before and after zoom. The gain should be positive and under about a point at 448. If it is not, stop and check before using the boxes.
- `fixed` and `broken` count the photos where zoom corrected or broke the answer.

In [ ]:
from IPython.display import Image as Show, display

for bucket in ("fixed_tight", "fixed_offcentre", "broken"):
    sheet = OUT / f"sheet_{bucket}.png"
    if sheet.is_file():
        print(bucket)
        display(Show(str(sheet)))

## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")